# Lab 1: Agents & Models

**Difficulty: Beginner | ~30 min | No prerequisites**

You will build a tiny AI **agent**, give it a tool, and swap the **model** underneath it — proving the model is a swappable part, not the whole system.

**In plain words:** a *model* is a program that talks — send it a message and it predicts a reply, but it can't do anything with your data. An *agent* wraps a model with **tools** (functions it can call) and a **loop** (machinery that runs the tools and feeds results back), so the model can actually *do* things. The model never runs your code — it describes what it wants, and the agent loop executes it. That's what you'll build here.

## Step 1

One command installs all required modules, versions pinned so the lab is reproducible.

In [1]:
# One command installs all required modules (versions pinned for reproducibility)
!pip install -qU "langchain==1.2.15" "langchain-core==1.2.28" "langchain-openai==1.1.12" "python-dotenv==1.2.2"


  Using cached langchain-1.2.15-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_core-1.2.28-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain_openai-1.1.12-py3-none-any.whl.metadata (3.1 kB)
  Using cached langgraph-1.1.10-py3-none-any.whl.metadata (8.0 kB)
INFO: pip is looking at multiple versions of langgraph to determine which version is compatible with other requirements. This could take a while.
  Using cached langgraph-1.1.9-py3-none-any.whl.metadata (8.0 kB)
  Using cached langgraph-1.1.8-py3-none-any.whl.metadata (8.0 kB)
  Using cached langgraph-1.1.6-py3-none-any.whl.metadata (8.0 kB)
  Using cached langgraph_checkpoint-4.1.1-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_prebuilt-1.0.13-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.3.15-py3-none-any.whl.metadata (1.7 kB)
  Using cached ormsgpack-1.12.2-cp311-cp311-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (3.2 kB)
INFO: pip is looking at 

## Step 2

Loads your OpenRouter API key from `.env` and stops with a clear message if it's missing.

In [2]:
import os
from dotenv import load_dotenv

# Read the OPENROUTER_API_KEY we saved in .env (Step 9 of the guide)
load_dotenv()

# Stop early with a clear message if the key is missing
if not os.getenv("OPENROUTER_API_KEY"):
    raise SystemExit("No OPENROUTER_API_KEY found. Add it to .env and restart the kernel.")

## Step 3

Create the agent's brain — a chat model pointed at the free `openai/gpt-oss-20b:free` model on OpenRouter.

In [3]:
from langchain_openai import ChatOpenAI

# Model 1: an open-weight model served free by OpenRouter
# base_url redirects the standard OpenAI client to OpenRouter
model_1 = ChatOpenAI(
    model="openai/gpt-oss-20b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0,
)

## Step 4

A tool is just a function with a docstring and type hints; LangChain reads those to tell the model what the tool does.

In [4]:
def multiply(a: float, b: float) -> float:
    """Multiply two numbers and return their product."""
    return a * b

## Step 5

Wrap the model and tools into the agent loop — `create_agent` is one line.

In [5]:
from langchain.agents import create_agent

# The agent = this model + this tool + the decision loop around them
agent = create_agent(model_1, tools=[multiply])

## Step 6

Run the agent: `invoke` runs the loop, and the last message in `messages` is the final answer.

In [6]:
# Run the agent loop with a user message
result_1 = agent.invoke({
    "messages": [("user", "What is 8 multiplied by 7?")],
})

# Show only the final answer, not the whole conversation
result_1["messages"][-1].content

'8 multiplied by 7 is **56**.'

## Step 7

Build a second agent with a different free model — same tool, same structure, only the model changed.

In [7]:
# Model 2: a different open-weight model, still free on OpenRouter
model_2 = ChatOpenAI(
    model="nvidia/nemotron-nano-9b-v2:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0,
)

# Same tools, same loop, new brain
agent_2 = create_agent(model_2, tools=[multiply])

## Step 8

Same question, same tool, same loop — only the model differs.

In [8]:
# Run the same question through the swapped model
result_2 = agent_2.invoke({
    "messages": [("user", "What is 8 multiplied by 7?")],
})
result_2["messages"][-1].content

'\n\nThe product of 8 and 7 is **56**.\n'

## Step 9

Agents don't always use tools; this question is answered directly from the model's knowledge.

In [9]:
# A plain-knowledge question: no tool, the loop just answers
result_3 = agent_2.invoke({
    "messages": [("user", "In one sentence, what is an AI agent?")],
})
result_3["messages"][-1].content

'\n\nAn AI agent is a system that autonomously performs tasks or makes decisions using artificial intelligence to perceive its environment and act upon it.\n'

## Optional Exercise

Swap in a third model. Add a new cell: create a `ChatOpenAI` with the free model `inclusionai/ling-3.0-flash:free`, build an `agent_3` with the same `multiply` tool, and ask it the same "8 multiplied by 7" question. Confirm it returns 56. If that model ID no longer exists, pick any `:free` model at https://openrouter.ai/models.